In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

/Users/cm/Library/Mobile Documents/com~apple~CloudDocs/文件/UCI Courses/275 NLP/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("datastax/linkedin_job_listings")
print("rows:", ds.num_rows)
print("cols:", len(ds.column_names))

rows: {'train': 123849}
cols: 1


In [3]:
df = ds["train"].to_pandas()[["job_id", "company_id", "title", "description"]]
df.head()

,job_id,company_id,title,description
0,921716,2774458.0,Marketing Coordinator,Job descriptionA leading real estate firm in N...
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ..."
2,10998357,64896719.0,Assitant Restaurant Manager,The National Exemplar is accepting application...
3,23221523,766262.0,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...


In [4]:
df.loc[:4, "description"].tolist()

['Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with o

In [5]:
clean_title = df["title"].str.lower().str.strip()
clean_title.head(10)
df["clean_title"] = clean_title
s = df["clean_title"].head(1000)
s.value_counts().loc[lambda x: x > 1].head(10)

clean_title
project manager                    6
account executive                  6
associate attorney                 6
administrative assistant           5
marketing manager                  4
construction project manager       4
staff accountant                   4
customer service representative    4
software engineer                  3
account manager                    3
Name: count, dtype: int64

In [6]:
df_has_company = df[df["company_id"].notna()].copy()
df_has_company_dedup = df_has_company.drop_duplicates(
    subset=["company_id", "clean_title"], keep="first"
)
print(
    len(df_has_company),
    "->",
    len(df_has_company_dedup),
    "removed= ",
    len(df_has_company) - len(df_has_company_dedup),
)
df_has_company.head()

122132 -> 95502 removed=  26630


,job_id,company_id,title,description,clean_title
0,921716,2774458.0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,marketing coordinator
2,10998357,64896719.0,Assitant Restaurant Manager,The National Exemplar is accepting application...,assitant restaurant manager
3,23221523,766262.0,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,senior elder law / trusts and estates associat...
5,91700727,1481176.0,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,economic development and planning intern
6,103254301,81942316.0,Producer,Company DescriptionRaw Cereal is a creative de...,producer


In [7]:
df_no_company = df[df["company_id"].isna()].copy()

In [8]:
df_no_company = df_no_company.drop_duplicates(subset=["description"], keep="first")
df_no_company = df_no_company[df_no_company["description"].notna()]
df_no_company.head()

,job_id,company_id,title,description,clean_title
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",mental health therapist/counselor
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,service technician
7,112576855,NaN,Building Engineer,Summary: Due to the pending retirement of our ...,building engineer
12,56482768,NaN,Appalachian Highlands Women's Business Center,FULL JOB DESCRIPTION – PROGRAM DIRECTOR Appala...,appalachian highlands women's business center
13,56924323,NaN,Structural Engineer,Universal Structural Engineers is seeking a st...,structural engineer


In [9]:
threshold = 0.95

texts = df_no_company["description"]
X = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(texts)

S = cosine_similarity(X)
ii, jj = np.where(np.triu(S, k=1) >= threshold)

to_drop = set(df_no_company.index[j] for j in jj)

sample = list(to_drop)[:5]
for idx in sample:
    print("-------------------------")
    print(df_no_company.loc[idx, "description"][:800])

-------------------------
Manufacturing Engineer- Medical Device manufacturing - contract opportunity- benefits offered
Provides engineering support to the Product Transfer team, which executes Manufacturing Transfers and critical supply continuity projects.Supports PPAP and associated activities with new or existing suppliers.Possesses very good communicational skills and actively develop relationships with key stakeholders.This is an individual contributor role that requires the use of judgement in applying professional expertise and is expected to work independently with minimal supervision.A relevant BS Degree in Engineering and 4+ years of experience in medical devices or other similar highly regulated industries is required.
-------------------------
OTTO Agencies, under the umbrella of LLOYD Agencies, are hiring for sales management professionals across the United States.

WE ARE ONLY HIRING CANDIDATES IN TX, FL, MI, KY, IN, IL, AND GA


Remote and On-Site Opportunities availabl

In [10]:
df_no_company_dedup = df_no_company.drop(index=list(to_drop))

print(
    len(df_no_company),
    "->",
    len(df_no_company_dedup),
    "removed=",
    len(df_no_company) - len(df_no_company_dedup),
)

1695 -> 1668 removed= 27


In [11]:
df_cleaned = pd.concat([df_has_company_dedup, df_no_company_dedup], ignore_index=True)
display(df_cleaned[df_cleaned["company_id"].isna()].head(2))
df_cleaned[df_cleaned["company_id"].notna()].head(2)

,job_id,company_id,title,description,clean_title
95502,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",mental health therapist/counselor
95503,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,service technician


,job_id,company_id,title,description,clean_title
0,921716,2774458.0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,marketing coordinator
1,10998357,64896719.0,Assitant Restaurant Manager,The National Exemplar is accepting application...,assitant restaurant manager


In [12]:
import sqlite3

db_path = "linkedin_jobs_cleaned.sqlite"
table_name = "job_listings_cleaned"
meta_table_name = "column_metadata"

column_info = {
    "column_name": ["job_id", "title", "company", "company_id","description"],
    "description": ["Identifier to job", "Title of the job", "The company who posts this job", "Company identifier","Job description"],
    "data_type": ["int", "str", "str", "float","str"]
}
df_meta = pd.DataFrame(column_info)

conn = sqlite3.connect(db_path)

df_cleaned.to_sql(table_name, conn, if_exists="replace", index=False)

df_meta.to_sql(meta_table_name, conn, if_exists="replace", index=False)

conn.close()